# AgroVision AI: 03 - Model Experimentation & Evaluation
**Project:** AI-Based Multi-Crop Yield Estimation for Tamil Nadu  
**Models:** Linear Regression (Baseline), Random Forest Regressor, XGBoost Regressor  
**Methodology:** Chronological Train/Validation/Test Split with Zero Leakage Preprocessing

---
### Key ML Design Rules:
1. **Chronological Splitting:** 2000–2016 (Train), 2017–2019 (Val), 2020–2022 (Test).
2. **Leakage Exclusion:** `production` is strictly excluded from features because $\text{Yield} = \frac{\text{Production} \times 1000}{\text{Area}}$.
3. **Metrics:** Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), and Coefficient of Determination ($R^2$).


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Add project root to path
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_preprocessing import load_dataset, clean_agricultural_data
from src.feature_engineering import identify_model_features, chronological_split, build_preprocessor
from src.evaluate import calculate_metrics, save_model_comparison
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb


### 1. Data Ingestion & Chronological Split
Partition data by year to prevent future climate/economic leakage into past models.


In [ ]:
data_path = os.path.join("..", "data", "processed", "des_cleaned_all_crops.csv")
if not os.path.exists(data_path):
    data_path = os.path.join("data", "processed", "des_cleaned_all_crops.csv")

clean_df, _ = clean_agricultural_data(load_dataset(data_path))
cat_feats, num_feats = identify_model_features(clean_df)
feature_cols = cat_feats + num_feats

train_df, val_df, test_df = chronological_split(clean_df, train_end_year=2016, val_end_year=2019)
print(f"Train records (2000-2016):      {len(train_df)}")
print(f"Validation records (2017-2019): {len(val_df)}")
print(f"Test records (2020-2022):       {len(test_df)}")

X_train, y_train = train_df[feature_cols], train_df['yield'].values
X_val, y_val = val_df[feature_cols], val_df['yield'].values
X_test, y_test = test_df[feature_cols], test_df['yield'].values


### 2. Preprocessing Pipeline
Fit ColumnTransformer strictly on the training set.


In [ ]:
preprocessor = build_preprocessor(cat_feats, num_feats)
X_train_trans = preprocessor.fit_transform(X_train)
X_val_trans = preprocessor.transform(X_val)
X_test_trans = preprocessor.transform(X_test)
print(f"Transformed Feature Vector Dimension: {X_train_trans.shape[1]}")


### 3. Model Training & Comparison
Train and benchmark Linear Regression, Random Forest, and XGBoost.


In [ ]:
models = {
    'Linear Regression (Baseline)': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42),
    'XGBoost Regressor': xgb.XGBRegressor(n_estimators=150, max_depth=5, learning_rate=0.05, random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_trans, y_train)
    val_preds = model.predict(X_val_trans)
    m = calculate_metrics(y_val, val_preds)
    results.append({
        'Model': name,
        'Val MAE (kg/ha)': m['mae_kg_ha'],
        'Val RMSE (kg/ha)': m['rmse_kg_ha'],
        'Val R2 Score': m['r2_score']
    })

comparison_df = pd.DataFrame(results).sort_values(by='Val RMSE (kg/ha)')
display(comparison_df)


### 4. Final Evaluation on Held-Out Test Set (2020–2022)
Evaluate the best model (Linear Regression) on unseen future test records.


In [ ]:
best_model = models['Linear Regression (Baseline)']
test_preds = best_model.predict(X_test_trans)
test_metrics = calculate_metrics(y_test, test_preds)

print("=== FINAL TEST EVALUATION (2020-2022) ===")
print(f"Test MAE:   {test_metrics['mae_kg_ha']:,.2f} kg/ha")
print(f"Test RMSE:  {test_metrics['rmse_kg_ha']:,.2f} kg/ha")
print(f"Test R2:    {test_metrics['r2_score']:.4f}")
print(f"Mean Yield: {test_metrics['mean_actual_kg_ha']:,.2f} kg/ha")
